### Configurando Ambiente

In [ ]:
# Antes de rodar esta célula, certifique-se de que o driver NVIDIA está instalado
# (verifique com "nvidia-smi" no terminal). O pip install do torch abaixo
# detecta automaticamente o CUDA disponível no sistema.
%pip install transformers accelerate bitsandbytes torch psutil pandas

### Teste do Modelo na CPU

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import time
import gc

model_name = "Qwen/Qwen2.5-7B"

print("Baixando e carregando o tokenizador")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizador carregado com sucesso")

def benchmark(model, prompt, n=3):
    tok = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.time()
    for _ in range(n):
        model.generate(**tok, max_new_tokens=50)
    
    tempo_medio_ms = (time.time() - t0) * 1000 / n
    tokens_por_segundo = (50 * n) / (time.time() - t0)
    return tempo_medio_ms, tokens_por_segundo

def gerar_texto(modelo, prompt, max_tokens=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    outputs = modelo.generate(**inputs, max_new_tokens=max_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt_1 = "Qual o resultado da derivada de 5x²?"
prompt_2 = "Sobre o que se trata o livro Anjos e Demônios, de Dan Brown?"

In [ ]:
print("--- BASELINE QUALITATIVO (FP16 na CPU) ---")
print("Como estamos usando apenas o processador, a geração levará vários minutos. A razão dessa escolha é que não temos VRAM suficiente rs.")

model_cpu = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="cpu",
    torch_dtype=torch.bfloat16
)

print("\n--- RESPOSTAS EM FP16 ---")

print("\nResposta 1:")
resp_fp16_1 = gerar_texto(model_cpu, prompt_1)

print("\nResposta 2:")
resp_fp16_2 = gerar_texto(model_cpu, prompt_2)

In [ ]:
print("Removendo o modelo da memória")
del model_cpu
gc.collect()
print("RAM liberada.")

## Teste do Modelo na GPU
O teste abaixo foi realizado na GPU para demonstrar na prática o OutOfMemoryError e a Quantização do modelo

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
import torch
import time
import gc
import psutil
import threading
from threading import Thread

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Baixando e carregando o tokenizador")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizador carregado com sucesso")

def get_model_size_gb(model):
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_bytes + buffer_bytes) / (1024**3)

# Benchmark exigido no trabalho
def benchmark(model, prompt, n=3, max_new_tokens=50):
    tok = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = tok.input_ids.shape[1]

    ttfts = []
    tempos_geracao = []
    tokens_gerados_total = 0

    for _ in range(n):
        streamer = TextIteratorStreamer(
            tokenizer, skip_prompt=True, skip_special_tokens=True
        )
        gen_kwargs = dict(**tok, max_new_tokens=max_new_tokens, streamer=streamer)

        torch.cuda.synchronize()
        t_inicio = time.time()

        thread = Thread(target=model.generate, kwargs=gen_kwargs)
        thread.start()

        t_primeiro_token = None
        n_tokens = 0
        for _texto in streamer:
            if t_primeiro_token is None:
                t_primeiro_token = time.time()
            n_tokens += 1

        thread.join()
        torch.cuda.synchronize()
        t_fim = time.time()

        ttfts.append(t_primeiro_token - t_inicio)
        tempos_geracao.append(t_fim - t_primeiro_token)
        tokens_gerados_total += (n_tokens - 1)

    ttft_medio = sum(ttfts) / n
    tempo_geracao_total = sum(tempos_geracao)

    return {
        "tamanho_gb": get_model_size_gb(model),
        "tokens_prompt": prompt_len,
        "ttft_ms": ttft_medio * 1000,
        "velocidade_leitura_prompt_tps": prompt_len / ttft_medio,
        "velocidade_geracao_tps": tokens_gerados_total / tempo_geracao_total if tempo_geracao_total > 0 else 0,
    }

def imprimir_resultados(nome, resultados):
    print(f"\n--- RESULTADOS {nome} ---")
    print(f"Tamanho do modelo: {resultados['tamanho_gb']:.2f} GB")
    print(f"Tokens do prompt: {resultados['tokens_prompt']}")
    print(f"Tempo até o primeiro token (TTFT): {resultados['ttft_ms']:.2f} ms")
    print(f"Velocidade de leitura do prompt: {resultados['velocidade_leitura_prompt_tps']:.2f} tokens/s")
    print(f"Velocidade de geração: {resultados['velocidade_geracao_tps']:.2f} tokens/s")

def gerar_texto_chat(modelo, prompt, max_tokens=250):
    mensagens = [
        {"role": "user", "content": prompt}
    ]
    text_prompt = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text_prompt, return_tensors="pt").to(modelo.device)
    outputs = modelo.generate(**inputs, max_new_tokens=max_tokens)
    input_length = inputs.input_ids.shape[1]
    resposta_limpa = outputs[0][input_length:]
    return tokenizer.decode(resposta_limpa, skip_special_tokens=True)

prompt_1 = "Qual o resultado da derivada de 5x²?"
prompt_2 = "Qual a estrutura de um struct em C?"


# ===================== MÉTRICAS DE RECURSOS =====================

class MonitorRecursos:
    """Monitora o pico de RAM e VRAM durante um bloco de teste."""
    def __init__(self, intervalo=0.05):
        self.intervalo = intervalo
        self.rodando = False
        self.pico_ram_gb = 0
        self.processo = psutil.Process()

    def _monitorar(self):
        while self.rodando:
            ram_atual = self.processo.memory_info().rss / (1024**3)
            if ram_atual > self.pico_ram_gb:
                self.pico_ram_gb = ram_atual
            time.sleep(self.intervalo)

    def iniciar(self):
        self.pico_ram_gb = self.processo.memory_info().rss / (1024**3)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        self.rodando = True
        self.thread = threading.Thread(target=self._monitorar, daemon=True)
        self.thread.start()

    def parar(self):
        self.rodando = False
        self.thread.join()
        pico_vram_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0
        return self.pico_ram_gb, pico_vram_gb


def medir_contexto_maximo(model, tokenizer, tamanhos=None, max_new_tokens=1):
    """Testa até qual tamanho de contexto o modelo consegue rodar sem estourar memória."""
    if tamanhos is None:
        tamanhos = [512, 1024, 2048, 4096, 8192, 16384, 32768]

    contexto_teorico = getattr(model.config, "max_position_embeddings", None)
    contexto_pratico = 0

    for tamanho in tamanhos:
        try:
            if model.device.type == "cuda":
                torch.cuda.empty_cache()
            input_ids = torch.randint(0, tokenizer.vocab_size, (1, tamanho)).to(model.device)
            attention_mask = torch.ones_like(input_ids)
            with torch.no_grad():
                model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens)
            contexto_pratico = tamanho
            print(f"  OK: {tamanho} tokens de contexto")
        except torch.cuda.OutOfMemoryError:
            print(f"  OOM (VRAM) em {tamanho} tokens de contexto")
            torch.cuda.empty_cache()
            break
        except RuntimeError as e:
            print(f"  Erro em {tamanho} tokens: {e}")
            break

    return {"contexto_teorico": contexto_teorico, "contexto_pratico_maximo": contexto_pratico}


def imprimir_recursos(nome, pico_ram_gb, pico_vram_gb, contexto):
    print(f"\n--- RECURSOS {nome} ---")
    print(f"Pico de RAM: {pico_ram_gb:.2f} GB")
    if pico_vram_gb > 0:
        print(f"Pico de VRAM: {pico_vram_gb:.2f} GB")
    print(f"Contexto máximo (config do modelo): {contexto['contexto_teorico']} tokens")
    print(f"Contexto máximo suportado no hardware: {contexto['contexto_pratico_maximo']} tokens")

#### FP16

In [ ]:
print("Carregando o modelo base em FP16 na GPU")
print("Aviso: Isso fará o download de aproximadamente 14 GB a 15 GB de arquivos")

# O device_map="auto" divide o modelo entre a VRAM e a RAM do sistema
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_name, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

print("Modelo FP16 carregado com sucesso! (Verifique o nvtop)")

In [ ]:
print("Iniciando o benchmark em FP16 na GPU")

monitor = MonitorRecursos()
monitor.iniciar()

# Benchmark com o prompt_1
resultados_p1 = benchmark(model_fp16, prompt_1, n=3)
imprimir_resultados("BASELINE PROMPT_1 (FP16)", resultados_p1)

# Benchmark com o prompt_2
resultados_p2 = benchmark(model_fp16, prompt_2, n=3)
imprimir_resultados("BASELINE PROMPT_2 (FP16)", resultados_p2)

pico_ram, pico_vram = monitor.parar()

print("\nMedindo contexto máximo (GPU FP16)")
contexto_fp16 = medir_contexto_maximo(model_fp16, tokenizer)

imprimir_recursos("GPU FP16", pico_ram, pico_vram, contexto_fp16)

resultados_fp16 = {
    "prompt_1": resultados_p1, "prompt_2": resultados_p2,
    "pico_ram": pico_ram, "pico_vram": pico_vram, "contexto": contexto_fp16
}

In [ ]:
import gc
print("Removendo o modelo FP16 memória")
try:
    del model_fp16
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("VRAM completamente liberada | Verificar no nvtop")

#### INT8

In [ ]:
print("Configurando a quantização para INT8")
quantization_config_int8 = BitsAndBytesConfig(
    load_in_8bit=True,
)

print("Carregando o modelo quantizado")
model_int8 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config_int8,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Modelo INT8 carregado com sucesso na VRAM")

In [ ]:

print("Iniciando o benchmark em INT8")

monitor = MonitorRecursos()
monitor.iniciar()

# 1. Throughput e Latência
resultados_p1 = benchmark(model_int8, prompt_1, n=3)
imprimir_resultados("DESEMPENHO PROMPT_1 (INT8)", resultados_p1)

resultados_p2 = benchmark(model_int8, prompt_2, n=3)
imprimir_resultados("DESEMPENHO PROMPT_2 (INT8)", resultados_p2)

# 2. Avaliação Qualitativa
print("\n--- AVALIAÇÃO QUALITATIVA ---")

print("\nGerando Resposta 1")
resposta_1 = gerar_texto_chat(model_int8, prompt_1)
print(f"Prompt 1: {prompt_1}")
print(f"Resposta:\n{resposta_1}\n")

print("\nGerando Resposta 2")
resposta_2 = gerar_texto_chat(model_int8, prompt_2)
print(f"Prompt 2: {prompt_2}")
print(f"Resposta:\n{resposta_2}\n")

pico_ram, pico_vram = monitor.parar()

print("\nMedindo contexto máximo (GPU INT8)")
contexto_int8 = medir_contexto_maximo(model_int8, tokenizer)

imprimir_recursos("GPU INT8", pico_ram, pico_vram, contexto_int8)

resultados_int8 = {
    "prompt_1": resultados_p1, "prompt_2": resultados_p2,
    "pico_ram": pico_ram, "pico_vram": pico_vram, "contexto": contexto_int8
}

In [ ]:
import gc

print("Removendo o modelo INT8 memória")
try:
    del model_int8
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("VRAM completamente liberada | Verificar no nvtop")

#### INT4

In [ ]:
print("Configurando a quantização para INT4")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Carregando o modelo quantizado")
model_int4 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Modelo INT4 carregado com sucesso na VRAM")

In [ ]:

print("Iniciando o benchmark em INT4")

monitor = MonitorRecursos()
monitor.iniciar()

# 1. Throughput e Latência
resultados_p1 = benchmark(model_int4, prompt_1, n=3)
imprimir_resultados("DESEMPENHO PROMPT_1 (INT4)", resultados_p1)

resultados_p2 = benchmark(model_int4, prompt_2, n=3)
imprimir_resultados("DESEMPENHO PROMPT_2 (INT4)", resultados_p2)

# 2. Avaliação Qualitativa
print("\n--- AVALIAÇÃO QUALITATIVA ---")

print("\nGerando Resposta 1")
resposta_1 = gerar_texto_chat(model_int4, prompt_1)
print(f"Prompt 1: {prompt_1}")
print(f"Resposta:\n{resposta_1}\n")

print("\nGerando Resposta 2")
resposta_2 = gerar_texto_chat(model_int4, prompt_2)
print(f"Prompt 2: {prompt_2}")
print(f"Resposta:\n{resposta_2}\n")

pico_ram, pico_vram = monitor.parar()

print("\nMedindo contexto máximo (GPU INT4)")
contexto_int4 = medir_contexto_maximo(model_int4, tokenizer)

imprimir_recursos("GPU INT4", pico_ram, pico_vram, contexto_int4)

resultados_int4 = {
    "prompt_1": resultados_p1, "prompt_2": resultados_p2,
    "pico_ram": pico_ram, "pico_vram": pico_vram, "contexto": contexto_int4
}

In [ ]:
import gc

print("Removendo o modelo INT4 memória")
try:
    del model_int4
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print("VRAM completamente liberada | Verificar no nvtop")

### Análise dos modelos

In [ ]:
import pandas as pd

def montar_linha(nome_modelo, resultados):
    # Faz média dos dois prompts para as métricas de velocidade/latência
    p1, p2 = resultados["prompt_1"], resultados["prompt_2"]
    return {
        "Modelo": nome_modelo,
        "Tamanho (GB)": p1["tamanho_gb"],
        "TTFT médio (ms)": (p1["ttft_ms"] + p2["ttft_ms"]) / 2,
        "Vel. leitura prompt (tok/s)": (p1["velocidade_leitura_prompt_tps"] + p2["velocidade_leitura_prompt_tps"]) / 2,
        "Vel. geração (tok/s)": (p1["velocidade_geracao_tps"] + p2["velocidade_geracao_tps"]) / 2,
        "Pico RAM (GB)": resultados["pico_ram"],
        "Pico VRAM (GB)": resultados["pico_vram"],
        "Contexto teórico (tokens)": resultados["contexto"]["contexto_teorico"],
        "Contexto prático máx. (tokens)": resultados["contexto"]["contexto_pratico_maximo"],
    }

tabela_comparativa = pd.DataFrame([
    montar_linha("FP16", resultados_fp16),
    montar_linha("INT4", resultados_int4),
    montar_linha("INT8", resultados_int8),
])

# Arredondar valores numéricos para melhor leitura
colunas_numericas = tabela_comparativa.select_dtypes(include="number").columns
tabela_comparativa[colunas_numericas] = tabela_comparativa[colunas_numericas].round(2)

print("--- TABELA COMPARATIVA FINAL ---")
display(tabela_comparativa)

# Opcional: salvar como CSV para anexar no relatório
tabela_comparativa.to_csv("comparativo_quantizacao.csv", index=False)
print("\nTabela salva em 'comparativo_quantizacao.csv'")